In [ ]:
!pip install -q -U google-genai pymupdf faiss-cpu numpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 783.6/783.6 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 72.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.6/240.6 kB 875.9 kB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.2 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.4 which is incompatible.


In [ ]:
from google import genai
import fitz
import faiss
import numpy as np
import textwrap
import re
from google.colab import files

In [29]:
# Gemini client
client = genai.Client(api_key="AQ.Ab8RN6ItenfDDQTrN9rXTUO5xf_NulVkJqx8FsPi_gUJyW4mQA")

# model for generation
GEN_MODEL = "gemini-2.5-flash"

# model for embedding
EMBED_MODEL = "gemini-embedding-001"

In [30]:
print("Upload your resume PDF")
uploaded = files.upload()
pdf_path = next(iter(uploaded))

Upload your resume PDF


Saving RASIKA D_22CS134.pdf to RASIKA D_22CS134 (2).pdf


In [31]:
# reading text from uploaded resume pdf
def extract_text_from_pdf(path):
    doc = fitz.open(path)
    text = ""
    for page in doc:
        text += page.get_text("text", sort=True) + "\n"
    return text

resume_text = extract_text_from_pdf(pdf_path)
print("Resume text extracted successfully.")

Resume text extracted successfully.


In [32]:
# splitting long resume text into smaller chunks
def split_text(text, chunk_size=700):
    return textwrap.wrap(text, width=chunk_size)

chunks = split_text(resume_text)
print(f"Resume split into {len(chunks)} chunks.")

Resume split into 4 chunks.


In [33]:
# converting text chunks into embeddings
def embed_text(text):
    response = client.models.embed_content(
        model=EMBED_MODEL,
        contents=text
    )
    return response.embeddings[0].values

print("Creating embeddings for resume chunks...")
embeddings = [embed_text(chunk) for chunk in chunks]
print("Embeddings created.")

Creating embeddings for resume chunks...
Embeddings created.


In [34]:
# storing embeddings in faiss for retrieval
dimension = len(embeddings[0])

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings, dtype="float32"))

print("FAISS index created successfully.")

FAISS index created successfully.


In [35]:
print("Paste the job description below.")
print("Type 'done' on a new line when finished.\n")

job_description = ""
while True:
    line = input()
    if line.strip().lower() == "done":
        break
    job_description += line + "\n"

print("Job description captured.")

Paste the job description below.
Type 'done' on a new line when finished.

Candidate must have obtained at least 75% marks (or 8.0 CGPA) in 10th, 12th / Diploma, Bachelor’s degree and Master’s degree (if applicable).   For those in the final year, offer will be subject to them scoring 75% (8.0 CGPA) in the qualifying degree in 2026 without any backlog.   *Technical Skills Candidate must have the knowledge of programming in C or C++ along with their application for implementing data structures and algorithms.
done
Job description captured.


In [36]:
# finding most relevant resume parts for the given job description
query_embedding = np.array([embed_text(job_description)], dtype="float32")

top_k = min(3, len(chunks))
distances, indices = index.search(query_embedding, top_k)

retrieved_chunks = [chunks[i] for i in indices[0]]
context = "\n".join(retrieved_chunks)

print("Relevant resume chunks retrieved.")

Relevant resume chunks retrieved.


In [37]:
# manual skill database for matching
SKILL_DB = [
    "python", "java", "c", "c++", "sql", "mysql",
    "html", "css", "javascript", "react", "node.js",
    "machine learning", "deep learning", "data analysis",
    "numpy", "pandas", "matplotlib", "nlp", "nltk", "spacy",
    "data structures", "algorithms", "git", "github",
    "firebase", "faiss", "problem solving", "communication",
    "teamwork", "salesforce", "apex", "lwc", "flow"
]

In [38]:
# extracting some useful details manually

def extract_email(text):
    match = re.search(r'[\w\.-]+@[\w\.-]+\.\w+', text)
    return match.group(0) if match else "Not found"

def extract_phone(text):
    match = re.search(r'(\+?\d[\d\-\s]{8,}\d)', text)
    return match.group(0) if match else "Not found"

def extract_skills(text, skill_db):
    text_lower = text.lower()
    found_skills = []
    for skill in skill_db:
        if skill.lower() in text_lower:
            found_skills.append(skill)
    return sorted(set(found_skills))

resume_email = extract_email(resume_text)
resume_phone = extract_phone(resume_text)

resume_skills = extract_skills(resume_text, SKILL_DB)
job_skills = extract_skills(job_description, SKILL_DB)

print("Email:", resume_email)
print("Phone:", resume_phone)
print("Resume skills extracted.")
print("Job skills extracted.")

Email: rasika8d8@gmail.com
Phone: 8015238124
Resume skills extracted.
Job skills extracted.


In [39]:
# manual comparison between resume and job skills
matched_skills = sorted(set(resume_skills) & set(job_skills))
missing_skills = sorted(set(job_skills) - set(resume_skills))

if len(job_skills) > 0:
    basic_score = round((len(matched_skills) / len(job_skills)) * 100, 2)
else:
    basic_score = 0.0

print("Matched skills:", matched_skills)
print("Missing skills:", missing_skills)
print("Basic score:", basic_score)

Matched skills: ['c']
Missing skills: ['algorithms', 'c++', 'data structures']
Basic score: 25.0


In [40]:
prompt = f"""
I built a resume-job matcher using Python.

Use the following retrieved resume context and computed results to write a simple report.

Resume context:
{context}

Job description:
{job_description}

Computed results:
Email: {resume_email}
Phone: {resume_phone}
Resume skills: {resume_skills}
Required job skills: {job_skills}
Matched skills: {matched_skills}
Missing skills: {missing_skills}
Basic score: {basic_score}

Write the output in this format:

Candidate Details:
Resume Skills:
Required Skills:
Matched Skills:
Missing Skills:
Match Score:
Suitability:
Suggestions:

Keep the language simple, clear, and professional.
Do not make it sound too AI-generated.
"""

In [41]:
response = client.models.generate_content(
    model=GEN_MODEL,
    contents=prompt
)

final_report = response.text
print("Final report generated.")

Final report generated.


In [42]:
print(final_report)

Candidate Details:
Email: rasika8d8@gmail.com
Phone: 8015238124
Name: RASIKA D

Resume Skills:
c, communication, css, data analysis, deep learning, flow, html, java, javascript, machine learning, mysql, python, sql

Required Skills:
algorithms, c, c++, data structures

Matched Skills:
c

Missing Skills:
algorithms, c++, data structures

Match Score:
25.0%

Suitability:
The candidate meets the academic percentage requirements with a BE CGPA of 8.85. However, there is a significant gap in the required technical skills for this role. While C is matched, proficiency in C++, Data Structures, and Algorithms, which are crucial for the position, is not evident from the resume.

Suggestions:
*   Strengthen knowledge and practical application of C++ for implementing Data Structures and Algorithms.
*   Consider relevant projects or certifications to demonstrate proficiency in Data Structures and Algorithms.
*   Update the resume to clearly highlight any existing experience or skills in C++, Data 